# Pipeline MSSR — Análisis cuantitativo de super-resolución

Notebook de ejecución del experimento **260327** (procarioplancton, INT/BOD × DAPI/SYBR).

Flujo: **0** conversión `.oib`→`.tif` → **1** segmentación → **2** MSSR → **3** parámetros → **4** estadística + **5** heatmaps.

Toda la lógica vive en el paquete `MSSR_PIPELINE/`; este notebook solo orquesta. Para cambiar parámetros se edita `config.py`, no las celdas.

**Orden de uso:** ejecutar las celdas de arriba a abajo. Las celdas 1–3 se corren una vez por sesión; las etapas 5–9 son el análisis.

## 1. Dependencias
Colab borra los paquetes al reiniciar la sesión: ejecutar esta celda **al inicio de cada sesión** (los archivos en Drive nunca se pierden, solo los paquetes).

In [ ]:
!pip install oiffile tifffile scikit-image opencv-python-headless scipy statsmodels seaborn --quiet
print("Dependencias instaladas.")

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Cargar el paquete (importación limpia)
Borra el caché (`__pycache__`) y descarga módulos viejos de memoria, de modo que siempre se ejecute la versión actual de los `.py` en Drive. Ejecutar tras cualquier cambio en los archivos del paquete.

Si moviste el paquete, ajusta `PKG`.

In [ ]:
import sys, shutil, os

PKG = "/content/drive/MyDrive/2025 Josue Villegas/LNMA 2026/MSSR_PIPELINE"

shutil.rmtree(os.path.join(PKG, "__pycache__"), ignore_errors=True)
for m in ["config", "mssr_core", "stage0_convert", "stage1_segmentation",
          "stage2_mssr", "stage3_parameters", "stage4_statistics",
          "stage5_heatmaps", "run_pipeline"]:
    sys.modules.pop(m, None)
if PKG not in sys.path:
    sys.path.insert(0, PKG)

import config as cfg
import stage0_convert as s0
import run_pipeline
print("Paquete cargado desde:", os.path.dirname(s0.__file__))
print("BASE_DIR    :", cfg.BASE_DIR)
print("RAW_OIB_DIR :", cfg.RAW_OIB_DIR)
print("Condiciones :", [cfg.label(c) for c in cfg.CONDITIONS])

## 4. Conversión `.oib` → `.tif` (Etapa 0)
Convierte los `.oib` crudos, conservando solo el canal de fluorescencia, y los archiva por condición en `BASE_DIR/<tratamiento>/RAW/<fluorocromo>/`.

**Primero un ensayo en seco** (no escribe nada): confirmar que las 4 condiciones suman 100, que nada queda “sin clasificar” y que el canal de fluorescencia es unánime.

In [ ]:
# Ensayo en seco
s0.convert_all(dry_run=True);

In [ ]:
# Conversión real (escribe los 100 .tif). Correr solo si el ensayo en seco fue correcto.
s0.convert_all(dry_run=False);

## 5. Segmentación y control de calidad (Etapa 1)
Segmenta sobre imagen con sustracción de fondo (los recortes guardados siguen siendo crudos), excluye imágenes de bajo SNR (`QC_MIN_SNR`) y aplica el filtro de objetos estricto. Revisar los conteos por célula, el resumen “conservadas X/Y imágenes” y los descartes por criterio.

In [ ]:
run_pipeline.main(["1"])

### 5b. Verificación visual de overlays (opcional pero recomendado)
Muestra unos overlays para confirmar que la segmentación marca células reales. Ajustar la condición a inspeccionar.

In [ ]:
import matplotlib.pyplot as plt
from skimage import io

cond = cfg.CONDITIONS[0]                       # ← cambia el índice para ver otra condición
carpeta = cfg.salida_dir(cond)
overlays = sorted(f for f in os.listdir(carpeta) if f.endswith("_overlay.png"))[:4]

if overlays:
    fig, axes = plt.subplots(1, len(overlays), figsize=(5 * len(overlays), 5))
    axes = [axes] if len(overlays) == 1 else axes
    for ax, f in zip(axes, overlays):
        ax.imshow(io.imread(os.path.join(carpeta, f)))
        ax.set_title(f.replace("_overlay.png", ""), fontsize=8)
        ax.axis("off")
    plt.suptitle(cfg.label(cond)); plt.tight_layout(); plt.show()
else:
    print("No hay overlays en", carpeta)

## 6. MSSR (Etapa 2)
Aplica MSSR a cada recorte. Es la etapa más lenta (depende del número total de recortes).

In [ ]:
run_pipeline.main(["2"])

## 7. Parámetros: FWHM, picos, vecino más cercano y tamaño pre/post (Etapa 3)
Genera `peaks_results.csv` y `cell_summary.csv` por condición. El resumen por célula incluye los diámetros (círculo equivalente y Feret) en RAW y MSSR.

In [ ]:
run_pipeline.main(["3"])

## 8. Estadística (Etapa 4)
Descriptivos, ANOVA factorial, tests pareados RAW vs MSSR, análisis sub-difractivo (Rayleigh) y figuras. Las figuras se guardan en `BASE_DIR/STATS`.

In [ ]:
run_pipeline.main(["4"])

## 9. Heatmaps (Etapa 5)
Mapas de intensidad MSSR con escala global común, por condición.

In [ ]:
run_pipeline.main(["5"])

---
## Utilidades

### Limpieza segura entre corridas
Borra recortes, MSSR y heatmaps **sin** tocar los `.tif` de entrada en `RAW/`. Usar antes de repetir desde la Etapa 1 con parámetros nuevos.

In [ ]:
for c in cfg.CONDITIONS:
    shutil.rmtree(cfg.salida_dir(c),   ignore_errors=True)   # Salida/ (recortes, overlays)
    shutil.rmtree(cfg.mssr_dir(c),     ignore_errors=True)   # MSSR/
    shutil.rmtree(cfg.heatmap_dir(c),  ignore_errors=True)   # HEATMAPS_MSSR/
print("Limpieza hecha; los .tif de RAW/ se conservan.")

### Ejecutar todo de corrido
Tras las celdas 1–4 (dependencias, Drive, paquete y conversión real), corre las etapas 1–5 en una sola llamada.

In [ ]:
run_pipeline.main(["1", "2", "3", "4", "5"])